# Nettoyage et préparation des données — Service Social OCP (2026)

Ce notebook réalise le nettoyage technique complet du dataset de réservations
(5000 lignes, 44 colonnes) avant la phase d'enrichissement et de modélisation
en schéma en étoile. Les règles métier ont déjà été validées à l'étape d'audit
précédente (39/39 tests, 0 anomalie) — ce notebook vérifie et corrige ici la
**qualité technique** : types, formats, cohérence interne, optimisation mémoire.

 Importation des bibliothèques

In [2]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", None)

## 1. Chargement du fichier CSV
Encodage `utf-8-sig` pour préserver les accents (le fichier source contient un BOM).

In [ ]:
df = pd.read_csv("dataset_ocp_social_2026_version2_brute.csv", encoding="utf-8-sig")
print(f"Fichier charge avec succes : {df.shape[0]} lignes, {df.shape[1]} colonnes")

Fichier charge avec succes : 5000 lignes, 44 colonnes


## 2. Aperçu général des données

In [4]:
df.head(5)
df.sample(5, random_state=42)

,numero_demande,site,matricule,nom_agent,prenom_agent,genre,situation_familiale,categorie_professionnelle,anciennete_annees,annees_depuis_dernier_sejour,nombre_points,type_produit,nom_etablissement,ville,id_periode,saison,date_demande,date_debut_sejour,date_fin_sejour,nombre_nuitees,periode_classement_debut,periode_classement_fin,date_limite_demande,date_limite_paiement,nombre_enfants,nombre_accompagnants,nombre_enfants_chambre_parents,total_membres_famille,nombre_chambres_double,nombre_chambres_simple,type_vue_config,formule_restauration,montant_total_sejour,quote_part_agent,montant_pris_charge_ocp,rang_classement,date_traitement,statut_apres_classement,reference_paiement,statut_paiement,date_paiement,statut_demande,numero_voucher,date_tirage_voucher
1501,CH100904,Khouribga,4447,Berrada,Badr,Homme,Marié,Employés,6,1,94,Hôtel-Club,Hôtel Kenzi Solazur,Tanger,P2,Basse,2026-01-08,2026-02-05,2026-02-13,8,2026-01-22,2026-01-29,2026-01-21,2026-02-02,2,1,1,4,1,1,2 chambre(s) + salon (4 pers) - Vue mer,All Inclusive,14377.93,2252.80,12125.13,6,2026-01-26,Acceptée,73195.0,Payé,2026-01-28,Confirmée,VOU-2026-01547,2026-01-29
2586,CH101992,Khouribga,7611,Tazi,Loubna,Femme,Célibataire,Employés,13,2,177,Résidence conventionnée,Résidence Malabata Bay,Tanger,P8,Haute,2026-05-27,2026-08-12,2026-08-22,10,2026-06-29,2026-07-03,2026-06-26,2026-07-06,0,1,0,2,0,1,1 chambre(s) + salon (2 pers) - Vue piscine,All Inclusive,6636.15,1725.82,4910.33,3,2026-07-03,Acceptée,71817.0,Payé,2026-07-06,Confirmée,VOU-2026-02633,2026-07-07
2653,CH103127,Khouribga,5659,Moutawakil,Souad,Femme,Mariée,Employés,4,5,218,Hôtel-Club,Hôtel Michliffen Ifrane,Ifrane,P2,Basse,2026-01-06,2026-02-05,2026-02-13,8,2026-01-22,2026-01-29,2026-01-21,2026-02-02,4,1,1,6,1,2,3 chambre(s) + salon (6 pers) - Vue standard,Hébergement seul,17417.58,3298.64,14118.94,4,2026-01-25,Acceptée,73909.0,Payé,2026-01-26,Confirmée,VOU-2026-01645,2026-01-29
1055,CH100489,Khouribga,5464,Ezzahi,Btissam,Femme,Célibataire,Agents de maîtrise et techniciens,1,4,165,Hôtel-Club,Hôtel Kenzi Menara Palace,Marrakech,P9,Haute,2026-05-27,2026-08-18,2026-08-24,6,2026-06-29,2026-07-03,2026-06-26,2026-07-06,0,0,0,1,0,1,1 chambre(s) + salon (1 pers) - Vue standard,All Inclusive,2194.74,539.83,1654.91,6,2026-07-01,Acceptée,44352.0,Payé,2026-07-02,Confirmée,VOU-2026-01525,2026-07-04
705,CH100378,Khouribga,3900,Ziani,Soukaina,Femme,Mariée,Employés,4,2,116,Hôtel-Club,Hôtel Mövenpick Malabata,Tanger,P5,Haute,2026-05-29,2026-07-25,2026-08-04,10,2026-06-29,2026-07-03,2026-06-26,2026-07-03,2,1,0,4,1,1,2 chambre(s) + salon (4 pers) - Vue piscine,Demi-pension,20554.51,4705.71,15848.80,4,2026-07-02,Refusée,NaN,Non payé,NaN,Refusée,NaN,NaN


## 3. Informations générales (shape, info, mémoire)

In [5]:
print("Dimensions :", df.shape)
print()
df.info()
print()
mem_avant = df.memory_usage(deep=True).sum()
print(f"Memoire utilisee (avant nettoyage) : {mem_avant/1024:.1f} Ko")

Dimensions : (5000, 44)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 44 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   numero_demande                  5000 non-null   object 
 1   site                            5000 non-null   object 
 2   matricule                       5000 non-null   int64  
 3   nom_agent                       5000 non-null   object 
 4   prenom_agent                    5000 non-null   object 
 5   genre                           5000 non-null   object 
 6   situation_familiale             5000 non-null   object 
 7   categorie_professionnelle       5000 non-null   object 
 8   anciennete_annees               5000 non-null   int64  
 9   annees_depuis_dernier_sejour    5000 non-null   int64  
 10  nombre_points                   5000 non-null   int64  
 11  type_produit                    5000 non-null   object 
 12  nom_etabl

## 4. Vérification des doublons
Contrôle qu'aucune ligne, aucune demande et aucun agent n'apparaît en double.

In [6]:
print("Lignes entierement dupliquees :", df.duplicated().sum())
print("Doublons numero_demande :", df["numero_demande"].duplicated().sum())
print("Doublons matricule :", df["matricule"].duplicated().sum())

Lignes entierement dupliquees : 0
Doublons numero_demande : 0
Doublons matricule : 650


## 5. Vérification des clés (numero_demande, matricule)
En plus de l'unicité, on vérifie que chaque identifiant est bien renseigné
(aucune valeur manquante) — une clé ne doit jamais être vide.

In [7]:
print("Valeurs manquantes numero_demande :", df["numero_demande"].isna().sum())
print("Valeurs manquantes matricule :", df["matricule"].isna().sum())
print("Nombre d'agents distincts (matricule) :", df["matricule"].nunique())
print("Nombre de demandes distinctes :", df["numero_demande"].nunique())

Valeurs manquantes numero_demande : 0
Valeurs manquantes matricule : 0
Nombre d'agents distincts (matricule) : 4350
Nombre de demandes distinctes : 5000


## 6. Vérification des valeurs manquantes
Les cases vides sur `reference_paiement`, `date_paiement`, `numero_voucher` et
`date_tirage_voucher` sont attendues : elles correspondent aux demandes non payées.
On vérifie que leur nombre correspond exactement au nombre de statuts "Non payé".

In [8]:
manquants = df.isna().sum()
print(manquants[manquants > 0])

print()
nb_non_paye = (df["statut_paiement"] == "Non payé").sum()
print(f"Nombre de demandes 'Non payé' : {nb_non_paye}")
print("Correspond aux valeurs manquantes ci-dessus ? ==>" ,
      (manquants[["reference_paiement","date_paiement","numero_voucher","date_tirage_voucher"]] == nb_non_paye).all())

reference_paiement     1829
date_paiement          1829
numero_voucher         1829
date_tirage_voucher    1829
dtype: int64

Nombre de demandes 'Non payé' : 1829
Correspond aux valeurs manquantes ci-dessus ? ==> True


## 7. Nettoyage des espaces et des chaînes de caractères
Suppression défensive des espaces superflus en début/fin de chaque colonne texte
(bonne pratique systématique, même si le fichier s'avère déjà propre).

In [9]:
colonnes_texte = df.select_dtypes(include="object").columns.tolist()

n_corrections = 0
for c in colonnes_texte:
    avant = df[c].copy()
    df[c] = df[c].str.strip()
    n_corrections += (avant != df[c]).sum()

print(f"Nombre de valeurs corrigees (espaces superflus) : {n_corrections}")

Nombre de valeurs corrigees (espaces superflus) : 5487


## 8. Conversion des types numériques
`reference_paiement` est un identifiant (5 chiffres), pas une valeur décimale.
A cause des cases vides, pandas l'avait stocke en `float64` (ex: 52075.0).
On le convertit en entier "nullable" (`Int64`), qui accepte les valeurs manquantes.

In [10]:
df["reference_paiement"] = df["reference_paiement"].astype("Int64")

print(df["reference_paiement"].dtype)
print(df["reference_paiement"].dropna().head(5).tolist())

Int64
[71260, 61783, 55790, 16988, 47271]


## 9. Conversion des colonnes de dates
Les 10 colonnes de dates sont converties du texte brut vers un vrai type `datetime`.

In [11]:
colonnes_dates = [
    "date_demande", "date_debut_sejour", "date_fin_sejour",
    "periode_classement_debut", "periode_classement_fin",
    "date_limite_demande", "date_limite_paiement",
    "date_traitement", "date_paiement", "date_tirage_voucher"
]

for c in colonnes_dates:
    df[c] = pd.to_datetime(df[c], errors="coerce")

print("=== Verification des types ===")
for c in colonnes_dates:
    ok = str(df[c].dtype) == "datetime64[ns]"
    print(f"[{'OK  ' if ok else 'ECHEC'}] {c:30s} -> {df[c].dtype}")

=== Verification des types ===
[OK  ] date_demande                   -> datetime64[ns]
[OK  ] date_debut_sejour              -> datetime64[ns]
[OK  ] date_fin_sejour                -> datetime64[ns]
[OK  ] periode_classement_debut       -> datetime64[ns]
[OK  ] periode_classement_fin         -> datetime64[ns]
[OK  ] date_limite_demande            -> datetime64[ns]
[OK  ] date_limite_paiement           -> datetime64[ns]
[OK  ] date_traitement                -> datetime64[ns]
[OK  ] date_paiement                  -> datetime64[ns]
[OK  ] date_tirage_voucher            -> datetime64[ns]


## 10. Vérification des formats des identifiants
`numero_demande` doit suivre le format CH + 6 chiffres, `matricule` doit être
un entier a 4 chiffres, `reference_paiement` un entier a 5 chiffres,
`numero_voucher` doit suivre le format VOU-2026-XXXXX.

In [13]:
print("numero_demande non conformes :",
      (~df["numero_demande"].str.match(r"^CH\d{6}$")).sum())

print("matricule hors plage [1000-9999] :",
      (~df["matricule"].between(1000, 9999)).sum())

ref_valides = df["reference_paiement"].dropna()
print("reference_paiement hors plage [10000-99999] :",
      (~ref_valides.between(10000, 99999)).sum())

vouchers_valides = df["numero_voucher"].dropna()
print("numero_voucher non conformes :",
      (~vouchers_valides.str.match(r"^VOU-2026-\d{5}$")).sum())

numero_demande non conformes : 0
matricule hors plage [1000-9999] : 0
reference_paiement hors plage [10000-99999] : 0
numero_voucher non conformes : 0


## 11. Vérification des domaines des variables catégorielles
Chaque variable catégorielle ne doit contenir que des valeurs attendues et
documentées (pas de faute de frappe, pas de valeur inconnue).

In [14]:
domaines_attendus = {
    "genre": {"Homme", "Femme"},
    "saison": {"Haute", "Basse"},
    "type_produit": {"Chalet OCP", "Résidence conventionnée", "Hôtel-Club"},
    "id_periode": {f"P{i}" for i in range(1, 10)},
    "statut_paiement": {"Payé", "Non payé"},
    "statut_demande": {"Confirmée", "Refusée"},
    "statut_apres_classement": {"Acceptée", "Refusée"},
    "formule_restauration": {"All Inclusive", "Demi-pension", "Hébergement seul"},
    "nombre_nuitees": {6, 8, 10},
}

for colonne, valeurs_ok in domaines_attendus.items():
    valeurs_reelles = set(df[colonne].unique())
    valeurs_inattendues = valeurs_reelles - valeurs_ok
    statut = "OK  " if not valeurs_inattendues else "ECHEC"
    print(f"[{statut}] {colonne:30s} -> valeurs inattendues : {valeurs_inattendues or 'aucune'}")

[OK  ] genre                          -> valeurs inattendues : aucune
[OK  ] saison                         -> valeurs inattendues : aucune
[OK  ] type_produit                   -> valeurs inattendues : aucune
[OK  ] id_periode                     -> valeurs inattendues : aucune
[OK  ] statut_paiement                -> valeurs inattendues : aucune
[OK  ] statut_demande                 -> valeurs inattendues : aucune
[OK  ] statut_apres_classement        -> valeurs inattendues : aucune
[OK  ] formule_restauration           -> valeurs inattendues : aucune
[OK  ] nombre_nuitees                 -> valeurs inattendues : aucune


## 12. Vérification des valeurs négatives
Aucune colonne numérique (montants, effectifs, ancienneté...) ne doit être négative.

In [15]:
colonnes_numeriques = df.select_dtypes(include=[np.number]).columns.tolist()

for c in colonnes_numeriques:
    n_negatifs = (df[c] < 0).sum()
    if n_negatifs > 0:
        print(f"[ECHEC] {c} : {n_negatifs} valeur(s) negative(s)")

print("Verification terminee : si rien n'est affiche au-dessus, aucune valeur negative.")

Verification terminee : si rien n'est affiche au-dessus, aucune valeur negative.


## 13. Vérification des dates incohérentes
Contrôle de la logique chronologique du processus : demande → classement →
traitement → paiement → voucher.

In [16]:
print("date_demande > date_limite_demande :",
      (df["date_demande"] > df["date_limite_demande"]).sum())

print("periode_classement_debut > periode_classement_fin :",
      (df["periode_classement_debut"] > df["periode_classement_fin"]).sum())

hors_fenetre = (df["date_traitement"] < df["periode_classement_debut"]) | \
               (df["date_traitement"] > df["periode_classement_fin"])
print("date_traitement hors fenetre de classement :", hors_fenetre.sum())

paye_mask = df["statut_paiement"] == "Payé"
paiement_incoherent = paye_mask & (
    (df["date_paiement"] < df["date_traitement"]) |
    (df["date_paiement"] > df["date_limite_paiement"])
)
print("date_paiement incoherente (hors fenetre autorisee) :", paiement_incoherent.sum())

voucher_incoherent = paye_mask & (df["date_tirage_voucher"] < df["date_paiement"])
print("date_tirage_voucher anterieure au paiement :", voucher_incoherent.sum())

date_demande > date_limite_demande : 0
periode_classement_debut > periode_classement_fin : 0
date_traitement hors fenetre de classement : 0
date_paiement incoherente (hors fenetre autorisee) : 0
date_tirage_voucher anterieure au paiement : 0


## 14. Vérification que nombre_nuitees = date_fin_sejour - date_debut_sejour

In [17]:
# Calcul de la durée réelle du séjour
duree_calculee = (df["date_fin_sejour"] - df["date_debut_sejour"]).dt.days

# Détection des incohérences
incoherences = df[duree_calculee != df["nombre_nuitees"]]

print("=" * 70)
print("VÉRIFICATION DE LA COHÉRENCE DU NOMBRE DE NUITÉES")
print("=" * 70)

if incoherences.empty:
    print("✅ Toutes les valeurs sont cohérentes.")
    print("Le nombre de nuitées correspond à la différence entre")
    print("'date_fin_sejour' et 'date_debut_sejour' pour toutes les demandes.")
else:
    print(f"❌ {len(incoherences)} incohérence(s) détectée(s).\n")

    display(
        incoherences[
            [
                "numero_demande",
                "date_debut_sejour",
                "date_fin_sejour",
                "nombre_nuitees"
            ]
        ].assign(duree_calculee=duree_calculee[incoherences.index])
    )

VÉRIFICATION DE LA COHÉRENCE DU NOMBRE DE NUITÉES
✅ Toutes les valeurs sont cohérentes.
Le nombre de nuitées correspond à la différence entre
'date_fin_sejour' et 'date_debut_sejour' pour toutes les demandes.


## 15. Vérification que quote_part_agent + montant_pris_charge_ocp = montant_total_sejour



In [18]:
# Vérification de la cohérence des montants
ecart = (
    df["montant_total_sejour"]
    - (df["quote_part_agent"] + df["montant_pris_charge_ocp"])
).abs()

# Lignes présentant une incohérence (tolérance de 0.01 DH)
incoherences = df[ecart > 0.01]

print("=" * 70)
print("VÉRIFICATION DE LA COHÉRENCE DES MONTANTS")
print("=" * 70)

if incoherences.empty:
    print("✅ Tous les montants sont cohérents.")
    print("Pour chaque demande :")
    print("montant_total_sejour = quote_part_agent + montant_pris_charge_ocp")
else:
    print(f"❌ {len(incoherences)} incohérence(s) détectée(s).\n")

    display(
        incoherences[
            [
                "numero_demande",
                "montant_total_sejour",
                "quote_part_agent",
                "montant_pris_charge_ocp"
            ]
        ].assign(ecart_DH=ecart[incoherences.index].round(2))
    )

VÉRIFICATION DE LA COHÉRENCE DES MONTANTS
✅ Tous les montants sont cohérents.
Pour chaque demande :
montant_total_sejour = quote_part_agent + montant_pris_charge_ocp


## 16. Vérification de la cohérence des statuts de paiement
Le workflow métier impose : Refusée au classement => jamais de paiement ;
Confirmée => forcement payee, avec voucher ; Refusée (finale) => jamais de voucher.

In [19]:
print("=" * 80)
print("VÉRIFICATION DE LA COHÉRENCE DES STATUTS MÉTIER")
print("=" * 80)

controles = {
    "Refusée au classement mais statut final différent de 'Refusée'":
        ((df["statut_apres_classement"] == "Refusée") &
         (df["statut_demande"] != "Refusée")),

    "Refusée au classement mais possède un voucher":
        ((df["statut_apres_classement"] == "Refusée") &
         (df["numero_voucher"].notna())),

    "Demande confirmée mais statut de paiement différent de 'Payé'":
        ((df["statut_demande"] == "Confirmée") &
         (df["statut_paiement"] != "Payé")),

    "Demande confirmée avec informations de paiement incomplètes":
        ((df["statut_demande"] == "Confirmée") &
         (
             df["numero_voucher"].isna() |
             df["reference_paiement"].isna() |
             df["date_paiement"].isna()
         )),

    "Demande refusée mais possède des informations de paiement":
        ((df["statut_demande"] == "Refusée") &
         (
             df["numero_voucher"].notna() |
             df["reference_paiement"].notna() |
             df["date_paiement"].notna()
         ))
}

for description, condition in controles.items():

    nb = condition.sum()

    if nb == 0:
        print(f"✅ {description}")
        print("   → Aucune incohérence détectée.\n")

    else:
        print(f"❌ {description}")
        print(f"   → {nb} incohérence(s) détectée(s).\n")

        display(
            df.loc[
                condition,
                [
                    "numero_demande",
                    "statut_apres_classement",
                    "statut_demande",
                    "statut_paiement",
                    "reference_paiement",
                    "numero_voucher",
                    "date_paiement"
                ]
            ]
        )

VÉRIFICATION DE LA COHÉRENCE DES STATUTS MÉTIER
✅ Refusée au classement mais statut final différent de 'Refusée'
   → Aucune incohérence détectée.

✅ Refusée au classement mais possède un voucher
   → Aucune incohérence détectée.

✅ Demande confirmée mais statut de paiement différent de 'Payé'
   → Aucune incohérence détectée.

✅ Demande confirmée avec informations de paiement incomplètes
   → Aucune incohérence détectée.

✅ Demande refusée mais possède des informations de paiement
   → Aucune incohérence détectée.



## 17. Extraction de type_vue_config
Le texte composite ("2 chambre(s) + salon (3 pers) - Vue standard") est conserve
pour l'affichage, mais on en extrait une colonne analytique propre `vue` et
`nombre_pieces_salon`, exploitables directement en filtre dans Power BI.

In [20]:
# ============================================================================
# Extraction des informations contenues dans la colonne 'type_vue_config'
# ============================================================================

# Extraction des différentes informations
extrait = df["type_vue_config"].str.extract(
    r"(?P<nb_chambres>\d+)\s+chambre\(s\)\s+\+\s+salon\s+\((?P<capacite_logement>\d+)\s+pers\)\s+-\s+(?P<type_vue>.+)"
)

# Nombre total de pièces = chambres + salon
df["nombre_pieces"] = extrait["nb_chambres"].astype("Int64") + 1

# Capacité d'hébergement
df["capacite_logement"] = extrait["capacite_logement"].astype("Int64")

# Type de vue
df["type_vue"] = extrait["type_vue"].str.strip()

print("=" * 80)
print("EXTRACTION DES INFORMATIONS DE LA COLONNE 'type_vue_config'")
print("=" * 80)

display(
    df[
        [
            "type_vue_config",
            "nombre_pieces",
            "capacite_logement",
            "type_vue"
        ]
    ].head()
)

EXTRACTION DES INFORMATIONS DE LA COLONNE 'type_vue_config'


,type_vue_config,nombre_pieces,capacite_logement,type_vue
0,2 chambre(s) + salon (3 pers) - Vue standard,3,3,Vue standard
1,1 chambre(s) + salon (2 pers) - Vue jardin,2,2,Vue jardin
2,1 chambre(s) + salon (1 pers) - Vue mer,2,1,Vue mer
3,2 chambre(s) + salon (3 pers) - Vue jardin,3,3,Vue jardin
4,2 chambre(s) + salon (3 pers) - Vue jardin,3,3,Vue jardin


## 18. Vérification de la cohérence avec total_membres_famille

In [21]:
# ============================================================================
# Vérification de la cohérence entre la capacité du logement
# et le nombre total des membres de la famille
# ============================================================================

incoherences = df[
    df["capacite_logement"] != df["total_membres_famille"]
]

print("=" * 80)
print("VÉRIFICATION DE LA COHÉRENCE DE LA CAPACITÉ DU LOGEMENT")
print("=" * 80)

if incoherences.empty:
    print("✅ Toutes les demandes sont cohérentes.")
    print("La capacité du logement correspond au nombre total des membres de la famille.")
else:
    print(f"❌ {len(incoherences)} incohérence(s) détectée(s).\n")

    display(
        incoherences[
            [
                "numero_demande",
                "type_vue_config",
                "capacite_logement",
                "total_membres_famille"
            ]
        ]
    )

VÉRIFICATION DE LA COHÉRENCE DE LA CAPACITÉ DU LOGEMENT
✅ Toutes les demandes sont cohérentes.
La capacité du logement correspond au nombre total des membres de la famille.


## 20. Résumé final de la qualité des données

In [23]:
print("="*80)
print("RESUME FINAL DU NETTOYAGE")
print("="*80)
print(f"Lignes                         : {df.shape[0]}")
print(f"Colonnes                       : {df.shape[1]}")
print(f"Doublons                       : {df.duplicated().sum()}")
print(f"Valeurs manquantes justifiees  : {df.isna().sum().sum()} (uniquement demandes non payees)")
print(f"Types corriges                 : dates -> datetime64, reference_paiement -> Int64")
print(f"Memoire finale                 : {df.memory_usage(deep=True).sum()/1024:.1f} Ko")
print("="*80)

RESUME FINAL DU NETTOYAGE
Lignes                         : 5000
Colonnes                       : 47
Doublons                       : 0
Valeurs manquantes justifiees  : 7316 (uniquement demandes non payees)
Types corriges                 : dates -> datetime64, reference_paiement -> Int64
Memoire finale                 : 8571.2 Ko


verification final

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 47 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   numero_demande                  5000 non-null   object        
 1   site                            5000 non-null   object        
 2   matricule                       5000 non-null   int64         
 3   nom_agent                       5000 non-null   object        
 4   prenom_agent                    5000 non-null   object        
 5   genre                           5000 non-null   object        
 6   situation_familiale             5000 non-null   object        
 7   categorie_professionnelle       5000 non-null   object        
 8   anciennete_annees               5000 non-null   int64         
 9   annees_depuis_dernier_sejour    5000 non-null   int64         
 10  nombre_points                   5000 non-null   int64         
 11  type

## 21. Sauvegarde du fichier nettoyé

In [25]:
# ============================================================================
# Sauvegarde du jeu de données nettoyé
# ============================================================================

nom_fichier = "dataset_ocp_social_2026_nettoye_finale_version2.csv"

df.to_csv(
    nom_fichier,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 80)
print("SAUVEGARDE DU JEU DE DONNÉES NETTOYÉ")
print("=" * 80)
print(f"✅ Le fichier a été enregistré avec succès : {nom_fichier}")
print(f"📄 Nombre de lignes   : {df.shape[0]}")
print(f"📄 Nombre de colonnes : {df.shape[1]}")

SAUVEGARDE DU JEU DE DONNÉES NETTOYÉ
✅ Le fichier a été enregistré avec succès : dataset_ocp_social_2026_nettoye_finale_version2.csv
📄 Nombre de lignes   : 5000
📄 Nombre de colonnes : 47
